# Reappraisal annotations: IRR & rater disagreement

In [ ]:
import os
import pandas as pd
import numpy as np
from irrCAC.raw import CAC
import matplotlib.pyplot as plt

In [ ]:
# make new repository to store IRR results
if not os.path.exists("IRR"):
    os.makedirs("IRR")

In [ ]:
import sys
print(sys.executable)

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")

In [ ]:
techniques = ['[GM]',  # growth mindset
              '[PR]',  # positive repurposing
              '[A]',  # aberration
              '[EJ]',  # external justification
              '[N]',  # normalizing
              '[UC]',  # unwarranted conclusions
              '[V]',  # validation
              '[H]']  # humor

techniques_long = ["growth mindset", "positive repurposing", "aberration", "external justification",
                   "normalizing", "unwarranted conclusions", "validation", "humor"]

In [ ]:
'''Old version retained for reference.
def calc_IRR(category, mode="gwet"):
    """ Calculates IRR for technique 'category' over all 20 annotation studies
        and returns list of 20 IRR coefficients. Set mode to either 'gwet' or
        'krip' to calculate Gwet's AC 1 and Krippendorff's alpha, respectively. """
    """ IRR = []

    # over all 20 studies...
    for i in range(1, 21):
        data = pd.read_csv(DATA_DIR / f"annotations-only-{technique}.csv", sep=";")  # load cleaned data
        data = data[[c for c in data.columns if category in c]]  # keep annotation columns only
        data = data.transpose()  # transpose: rows = questions, columns = raters

        # sanity check: entries in dataframe = 30 or 31?
        if len(data) != 30 and len(data) != 31:
            raise Exception("Number of rows in df is not 30 or 31.")

        # calculate IRR...
        cac = CAC(data)

        if mode == "gwet":  # Gwet's AC 1
            IRR.append(cac.gwet()['est']['coefficient_value'])
        if mode == "krip":  # Krippendorff's alpha
            IRR.append(cac.krippendorff()['est']['coefficient_value'])

    return IRR
'''

In [ ]:
def calc_IRR(category, mode="gwet"): #updated version using all the 606 data points instead of for each data batch like in og version
    df = pd.read_csv(DATA_DIR / f"annotations-only-{category}.csv", sep=";").astype("Int64")
    data = df.transpose()  # rows = questions, columns = raters

    cac = CAC(data)

    if mode == "gwet":
        return [cac.gwet()["est"]["coefficient_value"]]
    if mode == "krip":
        return [cac.krippendorff()["est"]["coefficient_value"]]

## Calculate & visualize Gwet's AC 1

In [ ]:
gwet = pd.DataFrame()

for technique in techniques:
    gwet[technique] = calc_IRR(technique, mode="gwet")

gwet.to_csv("IRR/gwets-ac-1.csv", sep=";")
gwet.head()

In [ ]:
print("-- Median Gwet's AC 1 over all 20 studies per reappraisal technique --")
for i, tech in enumerate(techniques):
    print(f"{techniques_long[i]}: {round(gwet[tech].median(), 3)}")

In [ ]:
y_pos = np.arange(len(techniques))

for i, col in zip(y_pos, techniques):
    x = gwet[col].values
    y = [i] * len(x)
    plt.scatter(x, y, alpha=0.3, color=plt.cm.Dark2.colors[i-1])
    plt.scatter(np.median(x), i, marker='x', color=plt.cm.Dark2.colors[i-1], s=80)

plt.yticks(y_pos, techniques_long, fontsize=8)
plt.xlabel("Gwet's AC 1")
plt.ylabel("Reappraisal Techniques")

plt.tight_layout()
plt.savefig("IRR/IRR-gwets.jpeg", dpi=300)

## Calculate & visualize Krippendorff's alpha

In [ ]:
krip = pd.DataFrame()

for technique in techniques:
    krip[technique] = calc_IRR(technique, mode="krip")

krip.to_csv("IRR/krippendorffs-alpha.csv", sep=";")
krip.head()

In [ ]:
print("-- Median Krippendorff's alpha over all 20 studies per reappraisal technique --")
for i, tech in enumerate(techniques):
    print(f"{techniques_long[i]}: {round(krip[tech].median(), 3)}")

In [ ]:
y_pos = np.arange(len(techniques))

for i, col in zip(y_pos, techniques):
    x = krip[col].values
    y = [i] * len(x)
    plt.scatter(x, y, alpha=0.3, color=plt.cm.Dark2.colors[i-1])
    plt.scatter(np.median(x), i, marker='x', color=plt.cm.Dark2.colors[i-1], s=80)

plt.yticks(y_pos, techniques_long, fontsize=8)
plt.xlabel("Krippendorff's Alpha")
plt.ylabel("Reappraisal Techniques")

plt.tight_layout()
plt.savefig("IRR/IRR-krip.jpeg", dpi=300)

## Measure decision quality (for binary labels)

In [ ]:
def count_decisions(technique):
    """ Counts unanimous, strong majority, weak majority, and split decisions for each of the 
        606 annotated reappraisals per 'technique' """
    # technique: "[GM]", "[PR]", "[A]", "[EJ]", "[N]", "[UC]", "[V]", "[H]"
    
    df = pd.read_csv(DATA_DIR / f"annotations-only-{technique}.csv", sep=";").astype("Int64")
    
    # [unanimous, strong_maj, weak_maj]
    overall = [0, 0, 0]
    positive = [0, 0, 0]
    negative = [0, 0, 0]
    
    for col in df.columns:
        frac = df[col].mean()
        if frac == 0:
            overall[0] += 1
            negative[0] += 1
        if frac == 1:
            overall[0] += 1
            positive[0] += 1
        if frac in (0.2, (1/6)):
            overall[1] += 1
            negative[1] += 1
        if frac in (0.8, (5/6)):
            overall[1] += 1
            positive[1] += 1
        if frac in (0.4, (2/6)):
            overall[2] += 1
            negative[2] += 1
        if frac in (0.6, (4/6)):
            overall[2] += 1
            positive[2] += 1
        # if frac == 0.5:
            # overall[3] +=1
            # negative[3] += 1
            # positive[3] += 1

    return overall, positive, negative

In [ ]:
def plot_decisions(ax, positive, negative, title):
    """ Plots distribution of decision quality per technique separately for positive (i.e. technique present)
        and negative decisions (i.e. technique absent) as counted in 'positive', 'negative' """
    # positive, negative: data, returned from count_decisions()
    # title: string of reappraisal technique, e.g. "Growth mindset"

    x = ["unanimous", "strong majority", "weak majority"]

    ax.bar(x, negative, color=plt.cm.Dark2.colors[0])
    ax.bar(x, positive, bottom=negative, color=plt.cm.Dark2.colors[1])

    ax.set_title(title)
    ax.tick_params(axis='x', labelsize=9)
    ax.set_ylim(0, 550)

In [ ]:
positives = [count_decisions(t)[1] for t in techniques]
negatives = [count_decisions(t)[2] for t in techniques]

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(16, 8))

for ax, pos, neg, title in zip(axs.flat, positives, negatives, techniques_long):
    plot_decisions(ax, pos, neg, title)

axs[0][3].legend(["negative", "positive"])
for i in range(4):
    axs[1][i].set_xlabel("Decision")
for j in range(2):
    axs[j][0].set_ylabel("Frequency")

fig.suptitle("Decision quality by reappraisal technique", fontsize=14)

plt.tight_layout()
plt.savefig("IRR/decision-quality.jpeg", dpi=300)

In [ ]:
print("--- Number of detected instances with weak majority threshold: ---")
for i, tech in enumerate(positives):
    print(f"{techniques_long[i]}: {np.sum(tech)}")

## Number of techniques per reappraisal

In [ ]:
def calculate_labels(mode="weak majority"):
    """ Calculate binary labels based on different thresholds:
        weak majority, strong majority, unanimous ('mode') """
    
    binary_techniques = pd.DataFrame(columns=techniques)

    if mode == "weak majority":
        thresholds = (1, 0.8, 0.6, 5/6, 4/6)
    if mode == "strong majority":
        thresholds = (1, 0.8, 5/6)
    if mode == "unanimous":
        thresholds = [1]
    
    for tech in techniques:
        df = pd.read_csv(DATA_DIR / f"annotations-only-{tech}.csv", sep=";").astype("Int64")
        binary_techniques[tech] = [1 if df[i].mean() in thresholds else 0 for i in df.columns]

    return binary_techniques

In [ ]:
def calculate_freq(mode="weak majority"):
    """ calculates number of reappraisals with different numbers of technqiues """
    labels = calculate_labels(mode=mode)
    counts = labels.sum(axis=1)
    
    freq = {}
    for i in range(counts.max() + 1):
        freq[i] = counts.to_list().count(i)

    return freq

In [ ]:
weak_freq = calculate_freq(mode="weak majority")
strong_freq = calculate_freq(mode="strong majority")
uni_freq = calculate_freq(mode="unanimous")

In [ ]:
# plot number of reappraisals with different numbers of techniques
# according to thresholds: weak majority, strong majority, unanimous
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].bar(weak_freq.keys(), weak_freq.values(), color=plt.cm.Dark2.colors[0])
axs[1].bar(strong_freq.keys(), strong_freq.values(), color=plt.cm.Dark2.colors[1])
axs[2].bar(uni_freq.keys(), uni_freq.values(), color=plt.cm.Dark2.colors[2])

for i in range(3):
    axs[i].set_ylim(0, 450)

axs[2].set_xticks([0, 1, 2])

axs[0].set_ylabel("Frequency")
axs[1].set_xlabel("Count")

axs[0].set_title("Weak majority", fontsize=10)
axs[1].set_title("Strong majority", fontsize=10)
axs[2].set_title("Unanimous", fontsize=10)

fig.suptitle("Techniques per reappraisal")

fig.tight_layout()
plt.savefig("IRR/techniques-per-reappraisal.jpeg", dpi=300)

## Drawing examples for few shot prompting

In [ ]:
labels_overall = pd.DataFrame()

labels = []
decision = []
question_codes = []

technique_dfs = {
    t: pd.read_csv(DATA_DIR / f"annotations-only-{t}.csv", sep=";").astype("Int64")
    for t in techniques
}

for q in technique_dfs[techniques[0]].columns:
    labels_temp = []
    decision_temp = []

    for t in techniques:
        frac = technique_dfs[t][q].mean()

        if frac == 1:
            labels_temp.append(t[1:-1])
            decision_temp.append("unanimous")
        elif frac in (0.8, 5/6):
            labels_temp.append(t[1:-1])
            decision_temp.append("strong maj")
        elif frac in (0.6, 4/6):
            labels_temp.append(t[1:-1])
            decision_temp.append("weak maj")

    question_codes.append(q)
    labels.append(labels_temp)
    decision.append(decision_temp)

labels_overall["question_code"] = question_codes
labels_overall["labels"] = labels
labels_overall["decision_quality"] = decision

In [ ]:
reapp_texts = pd.read_csv(DATA_DIR / "annotate-lime.csv")

labels_overall["q_number"] = labels_overall["question_code"].str[1:].astype(int)
labels_overall = labels_overall.sort_values("q_number").reset_index(drop=True)

labels_overall["reapp_text"] = reapp_texts["REAPPRAISAL"]
labels_overall["situation"] = reapp_texts["SITUATION"]

labels_overall

In [ ]:
labels_overall["q_number"] = [int(x[1:]) for x in labels_overall.question_code]
labels_overall = labels_overall.sort_values(by="q_number").reset_index().drop(columns=["q_number", "index"])

In [ ]:
def filter_decisions(decision_list, quality="unanimous"):
    try:
        return [i for i, j in enumerate(decision_list) if j == quality]
    except ValueError:
        return None

def filter_dataframe(df, quality="unanimous"):
    filter = [filter_decisions(x, quality) for x in df.decision_quality]
    drop_indices = [i for i, j in enumerate(filter) if j == []]
    decision_indices = [x for x in filter if x != []]
    
    reapp_filtered = df.drop(index=drop_indices)
    
    reapp_filtered["labels"] = [
        [labels[i] for i in idx]
        for labels, idx in zip(reapp_filtered["labels"], decision_indices)
    ]
    reapp_filtered = reapp_filtered.reset_index().drop(columns=["decision_quality", "index"])

    return reapp_filtered

In [ ]:
reapp_unanimous = filter_dataframe(labels_overall, "unanimous")
reapp_strong_maj = filter_dataframe(labels_overall, "strong maj")
reapp_weak_maj = filter_dataframe(labels_overall, "weak maj")

In [ ]:
def draw_prompt_examples(df):
    ind = []
    single_labels = df.drop(index=[i for i, j in enumerate(df.labels) if len(j) != 1])
    single_labels["labels"] = [x[0] for x in single_labels.labels]
    
    for t in techniques:
        random_ex = single_labels[single_labels.labels == t[1:-1]].sample(n=1, random_state=15)
        print(f"RANDOM EXAMPLE {t}:")
        print(random_ex.reapp_text.iloc[0])
        print("SITUATION:")
        print(random_ex.situation.iloc[0])
        
        ind.append(random_ex.question_code.iloc[0])

    return ind

In [ ]:
print("--- DRAWING UNANIMOUS PROMPT EXAMPLES ---\n")
ex_unanimous = draw_prompt_examples(reapp_unanimous)

In [ ]:
print("--- DRAWING STRONG MAJORITY PROMPT EXAMPLES ---\n")
ex_strong = draw_prompt_examples(reapp_strong_maj)

In [ ]:
print("--- DRAWING WEAK MAJORITY PROMPT EXAMPLES ---\n")
ex_weak = draw_prompt_examples(reapp_weak_maj)

In [ ]:
test_set = labels_overall.copy()
test_set = test_set[~test_set.question_code.isin(ex_unanimous + ex_strong + ex_weak)].reset_index().drop(columns="index")
test_set

In [ ]:
test_set_small = pd.DataFrame()

for t in techniques:
    if t == "[A]":
        sample = 8
    else:
        sample = 10

    temp = test_set.drop(index=[i for i, j in enumerate(test_set.labels) if t[1:-1] not in j]
                        ).sample(n=sample, random_state=15).drop(columns="decision_quality")
    temp["labels"] = [t[1:-1]] * sample
    test_set_small = pd.concat([test_set_small, temp])

test_set_small = test_set_small.sample(frac=1).reset_index().drop(columns="index")

In [ ]:
test_set_small.head()

In [ ]:
test_set_small.to_csv("test_set_small_LLM.csv", index=False, sep=";")